In [9]:
import os, re
from pathlib import Path
import numpy as np
import pandas as pd

import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [10]:
def load_bhsig(root_path, script_type):
    """
    root_path: folder path to BHSig260-Hindi or BHSig260-Bengali
    script_type: "H" or "B"
    Output schema:
      dataset, writer_id (string), sample_id (int), label (genuine/forgery), path
    """
    records = []
    for writer_folder in sorted(os.listdir(root_path)):
        writer_path = os.path.join(root_path, writer_folder)
        if not os.path.isdir(writer_path):
            continue

        for fname in os.listdir(writer_path):
            if not fname.lower().endswith(".tif"):
                continue

            # Example: H-S-160-F-16.tif
            parts = fname.split("-")
            if len(parts) < 5:
                continue

            writer_num = parts[2]          # "160"
            gf = parts[3].upper()          # "G" or "F"
            sample_str = parts[4].split(".")[0]  # "16"

            label = "genuine" if gf == "G" else "forgery"

            records.append({
                "dataset": f"BHSIG_{script_type}",
                "writer_id": f"BHSIG_{script_type}_{int(writer_num):03d}",  # e.g., BHSIG_H_160
                "sample_id": int(sample_str),
                "label": label,
                "path": os.path.join(writer_path, fname)
            })

    return pd.DataFrame(records)

bengali_path = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\BHSig260\BHSig260-Bengali"
hindi_path   = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\BHSig260\BHSig260-Hindi"

valid_bhsig_b = load_bhsig(bengali_path, "B")
valid_bhsig_h = load_bhsig(hindi_path, "H")

valid_bhsig = pd.concat([valid_bhsig_b, valid_bhsig_h], ignore_index=True)

print("Total images:", len(valid_bhsig))
print(valid_bhsig["label"].value_counts())
print("Unique writers:", valid_bhsig["writer_id"].nunique())
display(valid_bhsig.head())

Total images: 14040
label
forgery    7800
genuine    6240
Name: count, dtype: int64
Unique writers: 260


,dataset,writer_id,sample_id,label,path
0,BHSIG_B,BHSIG_B_001,1,forgery,W:\SRH study\Case Study 2\Offline Signature Ve...
1,BHSIG_B,BHSIG_B_001,2,forgery,W:\SRH study\Case Study 2\Offline Signature Ve...
2,BHSIG_B,BHSIG_B_001,3,forgery,W:\SRH study\Case Study 2\Offline Signature Ve...
3,BHSIG_B,BHSIG_B_001,4,forgery,W:\SRH study\Case Study 2\Offline Signature Ve...
4,BHSIG_B,BHSIG_B_001,5,forgery,W:\SRH study\Case Study 2\Offline Signature Ve...


In [11]:
def split_writers_any(valid_df, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    writers = np.array(sorted(valid_df["writer_id"].unique()))
    rng = np.random.default_rng(seed)
    rng.shuffle(writers)

    n = len(writers)
    n_train = int(round(n * train_ratio))
    n_val   = int(round(n * val_ratio))

    train_w = set(writers[:n_train])
    val_w   = set(writers[n_train:n_train+n_val])
    test_w  = set(writers[n_train+n_val:])

    return train_w, val_w, test_w

train_writers, val_writers, test_writers = split_writers_any(valid_bhsig, seed=42)

print("Train writers:", len(train_writers))
print("Val writers:", len(val_writers))
print("Test writers:", len(test_writers))
print("Overlap train-val:", len(train_writers & val_writers))
print("Overlap train-test:", len(train_writers & test_writers))
print("Overlap val-test:", len(val_writers & test_writers))

Train writers: 182
Val writers: 39
Test writers: 39
Overlap train-val: 0
Overlap train-test: 0
Overlap val-test: 0


In [12]:
def build_pools(df_subset):
    genuine_by_writer = {}
    forgery_by_writer = {}
    for wid, group in df_subset.groupby("writer_id"):
        g = group[group["label"] == "genuine"]["path"].tolist()
        f = group[group["label"] == "forgery"]["path"].tolist()
        if len(g) > 0: genuine_by_writer[wid] = g
        if len(f) > 0: forgery_by_writer[wid] = f
    return genuine_by_writer, forgery_by_writer

def generate_pairs_for_writers(valid_df, writer_set, n_pairs=60000, seed=42, neg_mix=0.9):
    """
    label=1: genuine-genuine same writer (positive)
    label=0: negative (mostly genuine-forgery same writer + some cross-writer genuine-genuine)
    neg_mix: fraction of negatives that are same-writer genuine-forgery
    """
    df_subset = valid_df[valid_df["writer_id"].isin(writer_set)].copy()
    genuine_by_writer, forgery_by_writer = build_pools(df_subset)

    writers = sorted(genuine_by_writer.keys())
    rng = np.random.default_rng(seed)

    n_pos = n_pairs // 2
    n_neg = n_pairs - n_pos
    n_neg_same  = int(round(n_neg * neg_mix))
    n_neg_cross = n_neg - n_neg_same

    writers_with_forg = sorted(set(genuine_by_writer) & set(forgery_by_writer))
    if len(writers) < 2:
        raise ValueError("Need >=2 writers for cross-writer negatives.")
    if len(writers_with_forg) == 0:
        raise ValueError("Need writers with both genuine and forgery samples.")

    pairs = []

    # positives
    for _ in range(n_pos):
        w = rng.choice(writers)
        g = genuine_by_writer[w]
        a, b = rng.choice(len(g), size=2, replace=False)
        pairs.append({"path_a": g[a], "path_b": g[b], "label": 1, "pair_type": "pos", "writer_a": w, "writer_b": w})

    # negatives same-writer (genuine vs forgery)
    for _ in range(n_neg_same):
        w = rng.choice(writers_with_forg)
        g = genuine_by_writer[w]
        f = forgery_by_writer[w]
        a = rng.integers(0, len(g))
        b = rng.integers(0, len(f))
        pairs.append({"path_a": g[a], "path_b": f[b], "label": 0, "pair_type": "neg_same_writer", "writer_a": w, "writer_b": w})

    # negatives cross-writer (genuine vs genuine)
    for _ in range(n_neg_cross):
        w1, w2 = rng.choice(writers, size=2, replace=False)
        g1 = genuine_by_writer[w1]
        g2 = genuine_by_writer[w2]
        a = rng.integers(0, len(g1))
        b = rng.integers(0, len(g2))
        pairs.append({"path_a": g1[a], "path_b": g2[b], "label": 0, "pair_type": "neg_cross_writer", "writer_a": w1, "writer_b": w2})

    return pd.DataFrame(pairs).sample(frac=1.0, random_state=seed).reset_index(drop=True)

train_pairs = generate_pairs_for_writers(valid_bhsig, train_writers, n_pairs=80000, seed=1, neg_mix=0.9)
val_pairs   = generate_pairs_for_writers(valid_bhsig, val_writers,   n_pairs=20000, seed=2, neg_mix=0.9)
test_pairs  = generate_pairs_for_writers(valid_bhsig, test_writers,  n_pairs=20000, seed=3, neg_mix=0.9)

print("Train pairs:", len(train_pairs), "Val pairs:", len(val_pairs), "Test pairs:", len(test_pairs))
print("Train label balance:\n", train_pairs["label"].value_counts(normalize=True))

Train pairs: 80000 Val pairs: 20000 Test pairs: 20000
Train label balance:
 label
1    0.5
0    0.5
Name: proportion, dtype: float64


In [13]:
IMG_SIZE = (224, 224)

def load_preprocess(path):
    path = path.numpy().decode("utf-8")
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)  # works for .tif
    if img is None:
        raise ValueError(f"Failed to read image: {path}")
    img = cv2.resize(img, IMG_SIZE)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=-1)
    return img

def tf_load_preprocess(path):
    img = tf.py_function(load_preprocess, [path], Tout=tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 1])
    return img

def make_pair_dataset(pairs_df, batch_size=32, shuffle=True):
    a_paths = pairs_df["path_a"].astype(str).values
    b_paths = pairs_df["path_b"].astype(str).values
    labels  = pairs_df["label"].astype(np.float32).values

    ds = tf.data.Dataset.from_tensor_slices((a_paths, b_paths, labels))

    def map_fn(a, b, y):
        img_a = tf_load_preprocess(a)
        img_b = tf_load_preprocess(b)
        return (img_a, img_b), y

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(4000, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_pair_dataset(train_pairs, batch_size=32, shuffle=True)
val_ds   = make_pair_dataset(val_pairs,   batch_size=32, shuffle=False)
test_ds  = make_pair_dataset(test_pairs,  batch_size=32, shuffle=False)

(batch_imgs, batch_y) = next(iter(train_ds))
print(batch_imgs[0].shape, batch_imgs[1].shape, batch_y.shape)
print("Label counts in batch:", np.unique(batch_y.numpy(), return_counts=True))

(32, 224, 224, 1) (32, 224, 224, 1) (32,)
Label counts in batch: (array([0., 1.], dtype=float32), array([17, 15]))


In [14]:
EMB_DIM = 128 

def build_embedding_mobilenetv2(input_shape=(224,224,1), emb_dim=128, backbone_trainable=False):
    inp = keras.Input(shape=input_shape, name="sig_in")

    # MobileNetV2 expects 3-channel input
    x = layers.Concatenate(name="gray_to_rgb")([inp, inp, inp])  # (H,W,3)

    backbone = keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",     # set None if you want from scratch
        input_tensor=x
    )
    backbone.trainable = backbone_trainable  # start frozen

    x = backbone.output
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    emb = layers.Dense(emb_dim, name="emb")(x)
    emb = layers.Lambda(lambda t: tf.nn.l2_normalize(t, axis=1), name="l2_norm")(emb)

    model = keras.Model(inp, emb, name="embedding_mobilenetv2")
    return model

embedding_net = build_embedding_mobilenetv2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 1),
    emb_dim=EMB_DIM,
    backbone_trainable=False
)
embedding_net.summary()

C:\Users\91900\AppData\Local\Temp\ipykernel_29036\3209341514.py:9: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone = keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 10s 1us/step


Model: "embedding_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sig_in (InputLayer) │ (None, 224, 224,  │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gray_to_rgb         │ (None, 224, 224,  │          0 │ sig_in[0][0],     │
│ (Concatenate)       │ 3)                │            │ sig_in[0][0],     │
│                     │                   │            │ sig_in[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ gray_to_rgb[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 2,618,816 (9.99 MB)

 Trainable params: 360,832 (1.38 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [15]:
class SquaredEuclidean(layers.Layer):
    def call(self, inputs):
        a, b = inputs
        return tf.reduce_sum(tf.square(a - b), axis=1, keepdims=True)

def build_siamese_model(embedding_model, input_shape=(224,224,1)):
    a = keras.Input(shape=input_shape, name="img_a")
    b = keras.Input(shape=input_shape, name="img_b")

    emb_a = embedding_model(a)
    emb_b = embedding_model(b)

    dist = SquaredEuclidean(name="sq_euclidean")([emb_a, emb_b])
    return keras.Model([a, b], dist, name="siamese_network")

siamese_model = build_siamese_model(
    embedding_net,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 1)
)
siamese_model.summary()

Model: "siamese_network"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img_a (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img_b (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_mobilene… │ (None, 128)       │  2,618,816 │ img_a[0][0],      │
│ (Functional)        │                   │            │ img_b[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sq_euclidean        │ (None, 1)         │          0 │ embedding_mobile… │
│ (SquaredEuclidean)  │                   │            │ embedding_mobile… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,618,816 (9.99 MB)

 Trainable params: 360,832 (1.38 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [16]:
class ContrastiveLoss(keras.losses.Loss):
    def __init__(self, margin=0.5, name="contrastive_loss"):
        super().__init__(name=name)
        self.margin = margin

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
        d = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
        pos = y_true * tf.square(d)
        neg = (1.0 - y_true) * tf.square(tf.maximum(self.margin - d, 0.0))
        return tf.reduce_mean(pos + neg)

loss_fn = ContrastiveLoss(margin=0.5)

save_dir = r"W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\BHSig260\improved_mobilenetv2"
os.makedirs(save_dir, exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        os.path.join(save_dir, "best_bhsig_mnv2_siamese.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

# -------------------------
# Stage 1: Train head (backbone frozen)
# -------------------------
siamese_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=loss_fn
)

history1 = siamese_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks
)

# -------------------------
# Stage 2: Fine-tune last ~30% of MobileNetV2
# -------------------------
# Find MobileNetV2 backbone inside embedding_net
backbone = None
for layer in embedding_net.layers:
    if isinstance(layer, keras.Model) and "mobilenetv2" in layer.name.lower():
        backbone = layer
        break

if backbone is None:
    raise RuntimeError("Could not find MobileNetV2 backbone inside embedding_net.")

backbone.trainable = True

# Freeze early layers, fine-tune later layers
n = len(backbone.layers)
for l in backbone.layers[: int(n * 0.7)]:
    l.trainable = False

siamese_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=loss_fn
)

history2 = siamese_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/8
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0374
Epoch 1: val_loss improved from None to 0.03004, saving model to W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\BHSig260\improved_mobilenetv2\best_bhsig_mnv2_siamese.keras

Epoch 1: finished saving model to W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\BHSig260\improved_mobilenetv2\best_bhsig_mnv2_siamese.keras
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 3110s 1s/step - loss: 0.0271 - val_loss: 0.0300 - learning_rate: 5.0000e-04
Epoch 2/8
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0158
Epoch 2: val_loss improved from 0.03004 to 0.02895, saving model to W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\BHSig260\improved_mobilenetv2\best_bhsig_mnv2_siamese.keras

Epoch 2: finished saving model to W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\BHSig260\improved_mobilenetv2\best_bhsig_mnv2_siamese.keras
2500/2500 ━━━━━━━━━━━━━━━━━━

RuntimeError: Could not find MobileNetV2 backbone inside embedding_net.

In [26]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
EMB_DIM = 128

class SquaredEuclidean(layers.Layer):
    def call(self, inputs):
        a, b = inputs
        return tf.reduce_sum(tf.square(a - b), axis=1, keepdims=True)

def build_embedding_mnv2_training_style(input_shape=(224,224,1), emb_dim=128, backbone_trainable=False):
    inp = keras.Input(shape=input_shape, name="sig_in")

    # SAME as training: grayscale -> 3ch with Concatenate layer
    x = layers.Concatenate(name="gray_to_rgb")([inp, inp, inp])  # (H,W,3)

    # CRITICAL: SAME as training: attach backbone using input_tensor=
    backbone = keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    backbone.trainable = backbone_trainable

    # SAME as training: use backbone.output
    x = backbone.output
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    emb = layers.Dense(emb_dim, name="emb")(x)

    # SAME as training, but add output_shape to avoid shape inference issues later
    emb = layers.Lambda(
        lambda t: tf.nn.l2_normalize(t, axis=1),
        output_shape=(emb_dim,),
        name="l2_norm"
    )(emb)

    model = keras.Model(inp, emb, name="embedding_mobilenetv2")
    return model

def build_siamese_model(embedding_model, input_shape=(224,224,1)):
    a = keras.Input(shape=input_shape, name="img_a")
    b = keras.Input(shape=input_shape, name="img_b")
    emb_a = embedding_model(a)
    emb_b = embedding_model(b)
    dist = SquaredEuclidean(name="sq_euclidean")([emb_a, emb_b])
    return keras.Model([a, b], dist, name="siamese_network")

embedding_net = build_embedding_mnv2_training_style(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 1),
    emb_dim=EMB_DIM,
    backbone_trainable=False
)

siamese_model = build_siamese_model(
    embedding_net,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 1)
)

print("✅ Rebuilt model in training style.")

C:\Users\91900\AppData\Local\Temp\ipykernel_29036\2358411332.py:20: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone = keras.applications.MobileNetV2(


✅ Rebuilt model in training style.


In [27]:
# weights_path should point to the extracted file, e.g. ...\model.weights.h5
siamese_model.load_weights(weights_path)
print("✅ Weights loaded successfully.")

✅ Weights loaded successfully.


In [28]:
def get_distances(model, ds):
    d_all, y_all = [], []
    for (x, y) in ds:
        d = model.predict(x, verbose=0).reshape(-1)
        d_all.append(d)
        y_all.append(y.numpy().reshape(-1))
    return np.concatenate(d_all), np.concatenate(y_all)

def compute_far_frr(d, y, thresholds):
    y = y.astype(int)
    neg = (y == 0)
    pos = (y == 1)
    far, frr = [], []
    for t in thresholds:
        far.append(np.mean(d[neg] <= t))
        frr.append(np.mean(d[pos] >  t))
    return np.array(far), np.array(frr)

val_d, val_y = get_distances(siamese_model, val_ds)
thresholds = np.quantile(val_d, np.linspace(0, 1, 1000))
thresholds = np.unique(thresholds)
far, frr = compute_far_frr(val_d, val_y, thresholds)

eer_idx = np.argmin(np.abs(far - frr))
eer = (far[eer_idx] + frr[eer_idx]) / 2
best_t = thresholds[eer_idx]

print("BHSig Validation EER:", float(eer))
print("Best threshold:", float(best_t))
print("FAR@EER:", float(far[eer_idx]), "FRR@EER:", float(frr[eer_idx]))

BHSig Validation EER: 0.154
Best threshold: 0.20198093858865407
FAR@EER: 0.1535 FRR@EER: 0.1545


In [29]:
test_d, test_y = get_distances(siamese_model, test_ds)
test_y = test_y.astype(int)

neg = (test_y == 0)
pos = (test_y == 1)

test_far = np.mean(test_d[neg] <= best_t)
test_frr = np.mean(test_d[pos] >  best_t)
test_avg = (test_far + test_frr) / 2

print("BHSig Test FAR (val threshold):", float(test_far))
print("BHSig Test FRR (val threshold):", float(test_frr))
print("BHSig Test avg error:", float(test_avg))

BHSig Test FAR (val threshold): 0.1143
BHSig Test FRR (val threshold): 0.1279
BHSig Test avg error: 0.12110000000000001


In [ ]:
# Save final models
siamese_model.save(os.path.join(save_dir, "bhsig_mnv2_siamese.keras"))
embedding_net.save(os.path.join(save_dir, "bhsig_mnv2_embedding.keras"))
